In [1]:
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import pandas as pd

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'

columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

/home/roben/Codes/PracticalDeepLearning/practicaldeeplearning/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv(url, names=columns)

In [3]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
import numpy as np

In [5]:
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness','Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

df.fillna(df.mean(), inplace=True)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.0,72.0,35.00000,155.548223,33.6,0.627,50,1
1,1,85.0,66.0,29.00000,155.548223,26.6,0.351,31,0
2,8,183.0,64.0,29.15342,155.548223,23.3,0.672,32,1
3,1,89.0,66.0,23.00000,94.000000,28.1,0.167,21,0
4,0,137.0,40.0,35.00000,168.000000,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101.0,76.0,48.00000,180.000000,32.9,0.171,63,0
764,2,122.0,70.0,27.00000,155.548223,36.8,0.340,27,0
765,5,121.0,72.0,23.00000,112.000000,26.2,0.245,30,0
766,1,126.0,60.0,29.15342,155.548223,30.1,0.349,47,1


In [6]:
df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [7]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scalar = StandardScaler()
X_train = scalar.fit_transform(X_train)
X_test = scalar.transform(X_test)

In [8]:
X_train.shape, X_test.shape

((537, 8), (231, 8))

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score

In [10]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())

study.optimize(objective, n_trials=50)

[I 2026-06-23 18:32:48,875] A new study created in memory with name: no-name-4da9bf22-4dfb-49a4-a65c-01d3e8e7373d
[I 2026-06-23 18:32:49,390] Trial 0 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 139, 'max_depth': 10}. Best is trial 0 with value: 0.7653631284916201.
[I 2026-06-23 18:32:49,901] Trial 1 finished with value: 0.7523277467411545 and parameters: {'n_estimators': 157, 'max_depth': 3}. Best is trial 0 with value: 0.7653631284916201.
[I 2026-06-23 18:32:50,374] Trial 2 finished with value: 0.7783985102420856 and parameters: {'n_estimators': 130, 'max_depth': 18}. Best is trial 2 with value: 0.7783985102420856.
[I 2026-06-23 18:32:50,571] Trial 3 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 52, 'max_depth': 14}. Best is trial 2 with value: 0.7783985102420856.
[I 2026-06-23 18:32:51,231] Trial 4 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 183, 'max_depth': 17}. Best is trial 2 with value: 0.778398

In [11]:
study.best_trial.value, study.best_trial.params

(0.7839851024208566, {'n_estimators': 73, 'max_depth': 18})

In [12]:
from sklearn.metrics import accuracy_score

best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

print(f"Test accuracy with best hyperparams: {test_accuracy:.2f}")

Test accuracy with best hyperparams: 0.76


In [13]:
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_contour, plot_param_importances

In [14]:
plot_optimization_history(study).show()

In [ ]:
# import sys
# !{sys.executable} -m pip install "nbformat>=4.2.0"

In [16]:
plot_parallel_coordinate(study).show()

In [17]:
plot_contour(study).show()

In [18]:
plot_param_importances(study).show()